# YOLO26 fine-tune on Colab — CVAT zip 업로드

Phase 1: 정지 차량 감지용 YOLO26(car 단일 클래스) fine-tune.

**라벨링: CVAT**. CVAT에서 bbox 라벨링 후 **YOLO 포맷으로 export → zip**을 받아
이 노트북에 업로드한다 (Drive 불필요).

사용 순서:
1. **런타임 → 런타임 유형 변경 → GPU (T4/L4)**
2. CVAT에서 car bbox 라벨링 → **Export → `YOLO 1.1`** (또는 `Ultralytics YOLO`) → zip 다운로드
3. 아래 **업로드 셀**에서 그 zip 업로드 (`data.yaml` + `images/` + `labels/` 구조)
4. `CONFIG` 채우고 위에서부터 실행
5. 끝나면 `best.pt`를 다운로드해서 `extract_labels.py --yolo_weights <best.pt>` 로 사용

> CVAT `YOLO 1.1` export는 `obj.data` / `obj.names` / `obj_train_data/` 구조,
> `Ultralytics YOLO` export는 `data.yaml` + `images/`+`labels/` 구조다. 아래 셀은
> 둘 다 처리하도록 `data.yaml`을 찾거나 없으면 생성한다.

## 1. 설치

In [ ]:
!pip install -q -U ultralytics   # YOLO26 지원 위해 최신 ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. CONFIG — 여기만 손대면 됨

CVAT export zip을 업로드(다음 섹션)하고, 학습 하이퍼파라미터만 여기서 조정.
Jetson 배포 입력 크기와 맞추려고 `IMGSZ=320`.

In [ ]:
MODEL    = 'yolo26n.pt'   # YOLO26: n / s / m / l / x (NMS-free, edge 최적화)
EPOCHS   = 100
IMGSZ    = 320            # Jetson 배포 입력 크기와 맞춤
BATCH    = 32             # T4 16GB 기준. OOM이면 16
PATIENCE = 30             # early-stop
RUN_NAME = 'car_v1'
OUTPUT_DIR = '/content/yolo_runs'
DATA_ROOT  = '/content/cvat_yolo'   # 업로드한 zip을 풀 위치

## 3. CVAT zip 업로드 + 압축해제

CVAT에서 받은 YOLO export zip을 업로드한다. `data.yaml`이 있으면 그대로 쓰고,
없으면(`YOLO 1.1` 구조) `obj.names`로부터 생성한다.

In [ ]:
import os, glob, zipfile, shutil, yaml
from google.colab import files

# 1) zip 업로드
shutil.rmtree(DATA_ROOT, ignore_errors=True)
os.makedirs(DATA_ROOT, exist_ok=True)
up = files.upload()                     # CVAT YOLO export zip 선택
zip_name = next(iter(up))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(DATA_ROOT)
print('extracted to', DATA_ROOT)

# 2) data.yaml 찾기 (Ultralytics YOLO export) — 없으면 YOLO 1.1 구조에서 생성
yamls = glob.glob(os.path.join(DATA_ROOT, '**', 'data.yaml'), recursive=True)
if yamls:
    DATA_YAML = yamls[0]
else:
    # CVAT 'YOLO 1.1': obj.names(클래스명) + obj_train_data/ (images+labels 혼재)
    names_files = glob.glob(os.path.join(DATA_ROOT, '**', 'obj.names'), recursive=True)
    assert names_files, 'data.yaml도 obj.names도 없음 — CVAT export 구조 확인'
    with open(names_files[0]) as f:
        names = [l.strip() for l in f if l.strip()]
    train_dir = os.path.dirname(names_files[0])
    img_dir = None
    for cand in ('obj_train_data', 'obj_Train_data', 'images'):
        d = os.path.join(train_dir, cand)
        if os.path.isdir(d):
            img_dir = d; break
    assert img_dir, 'YOLO 1.1 이미지 폴더(obj_train_data 등)를 못 찾음'
    DATA_YAML = os.path.join(DATA_ROOT, 'data.yaml')
    with open(DATA_YAML, 'w') as f:
        yaml.safe_dump({'path': train_dir, 'train': img_dir, 'val': img_dir,
                        'nc': len(names), 'names': names}, f)
    print('generated data.yaml (train=val: 소량이면 split 없이 동일 폴더 사용)')

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print('DATA_YAML:', DATA_YAML)
print('nc   :', cfg.get('nc'))
print('names:', cfg.get('names'))   # car 단일 클래스 기대

## 4. 학습

폐쇄 환경 + 단일 클래스라 mosaic는 약하게, flip은 끔(방향 정보 보존).
결과는 `OUTPUT_DIR/RUN_NAME/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    mosaic=0.5,
    mixup=0.0,
    fliplr=0.0,   # 좌우 flip 금지 (방향 정보)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
)

## 5. 검증

In [ ]:
best_pt = os.path.join(OUTPUT_DIR, RUN_NAME, 'weights', 'best.pt')
print('best.pt:', best_pt, 'exists=', os.path.exists(best_pt))

best = YOLO(best_pt)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, split='val')
print('mAP50   :', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))
print('names   :', best.names)

## 6. best.pt 다운로드

Drive 안 쓰므로 브라우저로 직접 다운로드. WSL에서 받아 `extract_labels.py --yolo_weights`로 사용.

In [ ]:
from google.colab import files
files.download(best_pt)

## 7. (선택) ONNX export

Jetson TRT engine용 중간 산출물. WSL standalone inference만이면 생략 가능.
TRT engine 빌드(trtexec)는 반드시 Jetson에서.

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
onnx_path = best_pt.replace('.pt', '.onnx')
print('ONNX:', onnx_path, 'exists=', os.path.exists(onnx_path))
files.download(onnx_path)